In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Reuse/cache controls — immediately after Drive mount.
REUSE_PREDICTIONS = True
RUN_MISSING_PREDICTIONS = True
FOLDS = [0, 1, 2, 3, 4]
MODEL_ZIP = '/content/drive/MyDrive/OpenPlaque/models/Dataset001_CCTA_DHM-20260703T233210Z-3-001.zip'
STUDY_ZIP = '/content/drive/MyDrive/OpenPlaque/Full_DICOM.zip'
ENSEMBLE_ROOT = '/content/drive/MyDrive/OpenPlaque/GPU_Plaque_5Fold_Ensemble_v1'
REPORT_ROOT = '/content/drive/MyDrive/OpenPlaque/LAD_Validated_Longitudinal_Plaque_Profile_Report'


# OpenPlaque — Validated LAD Longitudinal Plaque Profile

This notebook characterizes plaque-model support **only along the frozen 24.996653-mm LAD**.

The prior curved LAD series is a nonspatial/rotation stack. Its longitudinal registration is usable, but angular/source-space registration failed conservative validation. Therefore this experiment reports longitudinal plaque support, attenuation composition, and fold uncertainty — **not anatomical plaque volume (mm³), plaque area (mm²), or TPV**.

The final accepted LAD is explicitly registered to the older centerline used in the curved-series fit before any plaque profile is produced.


In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --depth 1 --branch lad-validated-longitudinal-plaque-profile-from-main https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%pip -q install nnunetv2 SimpleITK pydicom pandas scipy matplotlib
import os, sys, shutil, zipfile, subprocess
from pathlib import Path
import numpy as np, pandas as pd, SimpleITK as sitk
sys.path.insert(0, '/content/OpenPlaque/src')
!git -C /content/OpenPlaque rev-parse HEAD
!nvidia-smi || true


In [ ]:
# Ensure the LAD 5-fold consensus exists. Reuse all valid cached predictions first.
from openplaque.study import OpenPlaqueStudy
from openplaque.lad_validated_longitudinal_plaque_profile import build_consensus_from_fold_masks

ensemble_root = Path(ENSEMBLE_ROOT)
lad_root = ensemble_root / 'LAD'
consensus_fp = lad_root / 'LAD_5fold_consensus.nii.gz'
disagreement_fp = lad_root / 'LAD_5fold_disagreement.nii.gz'

if not (consensus_fp.exists() and disagreement_fp.exists()):
    built = build_consensus_from_fold_masks(Path('/content/drive/MyDrive/OpenPlaque'))
    if built[0] is not None:
        consensus_fp, disagreement_fp = built

missing_folds = [f for f in FOLDS if not (lad_root / f'fold_{f}' / 'LAD.nii.gz').exists()]
if not (consensus_fp.exists() and disagreement_fp.exists()) and missing_folds:
    if not RUN_MISSING_PREDICTIONS:
        raise FileNotFoundError(f'Missing LAD fold predictions: {missing_folds}')
    os.environ['nnUNet_raw'] = '/content/nnUNet_raw'
    os.environ['nnUNet_preprocessed'] = '/content/nnUNet_preprocessed'
    os.environ['nnUNet_results'] = '/content/nnUNet_results'
    for p in [os.environ['nnUNet_raw'], os.environ['nnUNet_preprocessed'], os.environ['nnUNet_results']]:
        Path(p).mkdir(parents=True, exist_ok=True)
    model_zip = Path(MODEL_ZIP)
    if not model_zip.exists():
        raise FileNotFoundError(model_zip)
    if not any(Path(os.environ['nnUNet_results']).rglob('checkpoint_final.pth')):
        with zipfile.ZipFile(model_zip) as z:
            z.extractall(os.environ['nnUNet_results'])

    study = OpenPlaqueStudy(STUDY_ZIP, extract_root='/content/full_dicom_lad_plaque_profile_prediction')
    image, _, _ = study.load_series(1043)
    work = Path('/content/lad_plaque_profile_prediction')
    inp = work / 'input'
    inp.mkdir(parents=True, exist_ok=True)
    sitk.WriteImage(image, str(inp / 'LAD_0000.nii.gz'))

    for fold in missing_folds:
        local = work / f'fold_{fold}'
        if local.exists():
            shutil.rmtree(local)
        local.mkdir(parents=True)
        cmd = ['nnUNetv2_predict', '-i', str(inp), '-o', str(local),
               '-d', 'Dataset001_CCTA_DHM', '-c', '3d_fullres', '-f', str(fold)]
        print('RUN MISSING FOLD:', ' '.join(cmd))
        subprocess.run(cmd, check=True)
        final_dir = lad_root / f'fold_{fold}'
        final_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(local / 'LAD.nii.gz', final_dir / 'LAD.nii.gz')

    built = build_consensus_from_fold_masks(Path('/content/drive/MyDrive/OpenPlaque'))
    consensus_fp, disagreement_fp = built

if not (consensus_fp.exists() and disagreement_fp.exists()):
    raise FileNotFoundError('Could not create LAD five-fold consensus/disagreement.')

print('Consensus:', consensus_fp)
print('Disagreement:', disagreement_fp)


In [ ]:
from IPython.display import display, Image
from openplaque.lad_validated_longitudinal_plaque_profile import (
    LADValidatedLongitudinalPlaqueProfileWorkflow,
    synthetic_longitudinal_plaque_profile_self_test,
)
test = synthetic_longitudinal_plaque_profile_self_test()
display(test)
assert test['passed'], test
wf = LADValidatedLongitudinalPlaqueProfileWorkflow(root='/content/drive/MyDrive/OpenPlaque')


In [ ]:
prov = wf.resolve_inputs()
mapping = wf.validate_centerline_overlap()
print('Input provenance:')
display(prov)
print('Accepted-LAD ↔ historical registration mapping:')
display(mapping)
print('Mapped accepted LAD curved interval:',
      mapping['curved_long_pixel_min'], 'to', mapping['curved_long_pixel_max'])


In [ ]:
summary = wf.build_profile()
print('STATUS:', summary['status'])
display(summary)
print('Longitudinal plaque-support zones:')
display(wf.support_zones)
print('1-mm longitudinal profile:')
display(wf.bin_profile)


In [ ]:
names = wf.make_figures()
for name in names:
    print(name)
    display(Image(filename=str(wf.out/name)))


In [ ]:
report, zip_path = wf.package()
print('STATUS:', wf.summary['status'])
print('ANATOMICAL PLAQUE VOLUME REPORTED:', wf.summary['anatomical_plaque_volume_mm3_reported'])
print('HTML:', report)
print('Final ZIP:', zip_path)
print('Drive search:')
print('https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_LAD_VALIDATED_LONGITUDINAL_PLAQUE_PROFILE_REPORT_BACK.zip')
